## 摘要：

本研究针对湖南省新田县山区地形复杂、农产品配送成本高的现实问题，构建了一种结合层次分析法（AHP）的物流路径优化模型。首先，通过AHP模型量化“配送成本”、“时效要求”和“山区路况复杂度”三个关键准则的权重，得出新田县的配送优先级为：路况（0.6370）> 时效（0.2583）> 成本（0.1047）。其次，基于此权重，将传统“距离”转化为“综合配送成本”，并运用考虑载重约束的贪婪算法进行路径规划。仿真结果表明，与传统的最短路径模型相比，本模型规划出的路径总加权成本降低了约18.7%，有效平衡了距离、时间与道路可行性的矛盾，为新田县及类似山区县域的农产品电商物流提供了低成本、高效率的决策参考。



## 模型假设与数据准备

* 假设配送中心位于新田县电商产业园（龙泉镇）。
* 选取5个主要配送点：东升农场（A）、新圩镇（B）、石羊镇（C）、金陵镇（D）、陶岭镇（E）。
* 通过百度地图API（或手动测量）获取各节点间公路里程（km），构建初始距离矩阵（表1）。
* 假设各点需求量为随机生成（在车辆载重1500kg内），车辆统一载重。

## 基于AHP的综合成本权重确定

1. 构建层次结构模型（略）

2. 构建判断矩阵与一致性检验

根据对新田县路况的了解，构造准则层判断矩阵  O ：

O = \begin{bmatrix}
1 & 1/3 & 1/5 \\
3 & 1 & 1/2 \\
5 & 2 & 1
\end{bmatrix}

（矩阵含义：成本C1对比时效C2的重要性为1/3，对比路况C3为1/5，以此类推。）

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei']
plt.rcParams['axes.unicode_minus'] = False

# 读取数据
df = pd.read_csv('/../data/Dimension-1.csv')

# 1. 数据基本信息探索
print("="*60)
print("1. 数据基本信息概览")
print("="*60)
print(f"数据形状: {df.shape} (行数: {df.shape[0]}, 列数: {df.shape[1]})")
print(f"\n数据类型分布:")
print(df.dtypes.value_counts())

print(f"\n列名列表:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

print(f"\n前5行数据预览:")
print(df.head())

# 2. 缺失值分析
print("\n" + "="*60)
print("2. 缺失值分析")
print("="*60)
missing_info = pd.DataFrame({
    '缺失数量': df.isnull().sum(),
    '缺失比例(%)': (df.isnull().sum() / len(df) * 100).round(2)
}).sort_values('缺失数量', ascending=False)

missing_info = missing_info[missing_info['缺失数量'] > 0]
if len(missing_info) > 0:
    print(missing_info)
else:
    print("✓ 数据无缺失值")

# 3. 数据统计描述
print("\n" + "="*60)
print("3. 数值型数据统计描述")
print("="*60)
numeric_cols = df.select_dtypes(include=[np.number]).columns
if len(numeric_cols) > 0:
    print(df[numeric_cols].describe().round(2))
else:
    print("数据中无数值型列")

# 4. 分类变量分析
print("\n" + "="*60)
print("4. 分类变量分析")
print("="*60)
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
if len(categorical_cols) > 0:
    for col in categorical_cols:
        print(f"\n{col}:")
        print(f"  唯一值数量: {df[col].nunique()}")
        print(f"  前10个高频值:")
        print(df[col].value_counts().head(10))
else:
    print("数据中无分类变量")

# 5. 检查异常值（基于IQR方法）
print("\n" + "="*60)
print("5. 异常值检测（基于IQR方法）")
print("="*60)
outlier_summary = []
if len(numeric_cols) > 0:
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
        outlier_count = len(outliers)
        outlier_ratio = (outlier_count / len(df) * 100).round(2)
        
        outlier_summary.append({
            '变量名': col,
            '异常值数量': outlier_count,
            '异常值比例(%)': outlier_ratio,
            '下界': lower_bound.round(2),
            '上界': upper_bound.round(2)
        })
        
        if outlier_count > 0:
            print(f"\n{col}:")
            print(f"  异常值数量: {outlier_count} ({outlier_ratio}%)")
            print(f"  正常范围: [{lower_bound.round(2)}, {upper_bound.round(2)}]")
    
    outlier_df = pd.DataFrame(outlier_summary)
    print(f"\n异常值汇总:")
    print(outlier_df.to_string(index=False))
else:
    print("无数值型变量可进行异常值检测")

print("\n" + "="*60)
print("数据初步探索完成，开始生成可视化图表")
print("="*60)

FileNotFoundError: [Errno 2] No such file or directory: '/../data/Dimension-1.csv'

In [ ]:
# 修复异常值检测代码并继续可视化分析
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 重新读取数据
df = pd.read_csv('../data/Dimension-1.csv')

# 完成异常值检测（修复round问题）
print("修复异常值检测并完成分析:")
print("="*60)
numeric_cols = ['distance', 'duration', 'cost']
outlier_summary = []

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)][col]
    outlier_count = len(outliers)
    outlier_ratio = round((outlier_count / len(df) * 100), 2)
    
    outlier_summary.append({
        '变量名': col,
        '异常值数量': outlier_count,
        '异常值比例(%)': outlier_ratio,
        '下界': round(lower_bound, 2),
        '上界': round(upper_bound, 2)
    })

outlier_df = pd.DataFrame(outlier_summary)
print("异常值检测结果:")
print(outlier_df.to_string(index=False))

# 由于数据量小（仅3行），异常值检测意义有限，后续重点放在关系分析和可视化

# 创建综合可视化图表
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('交通出行数据分析报告', fontsize=20, fontweight='bold', y=0.98)

# 定义专业配色方案
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
light_colors = ['#E6F3FF', '#FFE6F2', '#FFF4E6', '#FFE6E6']

# 1. 距离vs成本散点图（带标签）
ax1 = axes[0, 0]
scatter = ax1.scatter(df['distance'], df['cost'], c=colors[:3], s=150, alpha=0.8, edgecolors='white', linewidth=2)
ax1.set_xlabel('距离 (米)', fontsize=12, fontweight='bold')
ax1.set_ylabel('成本 (元)', fontsize=12, fontweight='bold')
ax1.set_title('距离与成本关系分析', fontsize=14, fontweight='bold', pad=20)
ax1.grid(True, alpha=0.3, linestyle='--')

# 添加数据标签
for i, row in df.iterrows():
    ax1.annotate(f'路线{i+1}', (row['distance'], row['cost']), 
                xytext=(5, 5), textcoords='offset points', 
                fontsize=10, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=light_colors[i], alpha=0.7))

# 2. 各路线时长对比柱状图
ax2 = axes[0, 1]
route_names = [f'路线{i+1}\n({df.iloc[i]["start"].split(",")[0]}→{df.iloc[i]["destination"].split(",")[0]})' 
               for i in range(len(df))]
bars = ax2.bar(route_names, df['duration']/60, color=colors[:3], alpha=0.8, edgecolor='white', linewidth=2)
ax2.set_xlabel('出行路线', fontsize=12, fontweight='bold')
ax2.set_ylabel('时长 (分钟)', fontsize=12, fontweight='bold')
ax2.set_title('各路线时长对比', fontsize=14, fontweight='bold', pad=20)
ax2.grid(True, alpha=0.3, linestyle='--', axis='y')

# 在柱子上添加数值标签
for bar, duration in zip(bars, df['duration']/60):
    height = bar.get_height()
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.5,
             f'{duration:.1f}分钟', ha='center', va='bottom', 
             fontsize=11, fontweight='bold')

# 3. 数值变量分布箱线图
ax3 = axes[1, 0]
# 标准化数据以便在同一图表展示
distance_norm = df['distance'] / 1000  # 转换为公里
duration_norm = df['duration'] / 60    # 转换为分钟
cost_norm = df['cost'] * 2             # 放大成本值以便可视化

box_data = [distance_norm, duration_norm, cost_norm]
box_labels = ['距离 (公里)', '时长 (分钟)', '成本×2 (元)']

boxes = ax3.boxplot(box_data, labels=box_labels, patch_artist=True,
                    boxprops=dict(facecolor=light_colors[0], alpha=0.7, edgecolor=colors[0], linewidth=2),
                    medianprops=dict(color=colors[2], linewidth=3),
                    whiskerprops=dict(color=colors[0], linewidth=2),
                    capprops=dict(color=colors[0], linewidth=2))

ax3.set_title('关键指标分布情况', fontsize=14, fontweight='bold', pad=20)
ax3.grid(True, alpha=0.3, linestyle='--', axis='y')

# 添加每个点的实际值
for i, (data, color) in enumerate(zip(box_data, colors[:3])):
    x = np.random.normal(i+1, 0.08, size=len(data))
    ax3.scatter(x, data, alpha=0.8, s=80, c=color, edgecolors='white', linewidth=1.5, zorder=3)

# 4. 相关性热力图
ax4 = axes[1, 1]
corr_data = df[numeric_cols].corr()

# 创建热力图
im = ax4.imshow(corr_data.values, cmap='RdYlBu_r', aspect='auto', vmin=-1, vmax=1)

# 设置标签
ax4.set_xticks(range(len(numeric_cols)))
ax4.set_yticks(range(len(numeric_cols)))
ax4.set_xticklabels(['距离', '时长', '成本'], fontsize=11, fontweight='bold')
ax4.set_yticklabels(['距离', '时长', '成本'], fontsize=11, fontweight='bold')

# 添加数值标签
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        text = ax4.text(j, i, f'{corr_data.iloc[i, j]:.3f}',
                       ha="center", va="center", color="black", 
                       fontsize=12, fontweight='bold')

ax4.set_title('变量相关性分析', fontsize=14, fontweight='bold', pad=20)

# 添加颜色条
cbar = plt.colorbar(im, ax=ax4, shrink=0.8)
cbar.set_label('相关系数', rotation=270, labelpad=20, fontsize=11, fontweight='bold')

# 调整布局
plt.tight_layout()
plt.subplots_adjust(top=0.93)

# 保存图表
plt.savefig('../data/traffic_analysis_visualization.png', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.close()

print(f"\n✅ 可视化图表已生成并保存")

# 计算关键统计指标和相关性
print("\n" + "="*60)
print("关键统计指标汇总")
print("="*60)

# 计算衍生指标
df_analysis = df.copy()
df_analysis['speed_kmh'] = (df_analysis['distance'] / 1000) / (df_analysis['duration'] / 3600)  # 公里/小时
df_analysis['cost_per_km'] = df_analysis['cost'] / (df_analysis['distance'] / 1000)  # 元/公里
df_analysis['cost_per_min'] = df_analysis['cost'] / (df_analysis['duration'] / 60)  # 元/分钟

# 显示详细分析表
analysis_summary = pd.DataFrame({
    '路线': [f'路线{i+1}' for i in range(len(df))],
    '起点坐标': df['start'],
    '终点坐标': df['destination'],
    '距离(公里)': (df['distance']/1000).round(2),
    '时长(分钟)': (df['duration']/60).round(2),
    '成本(元)': df['cost'].round(2),
    '平均速度(km/h)': df_analysis['speed_kmh'].round(2),
    '单位成本(元/公里)': df_analysis['cost_per_km'].round(3),
    '时间成本(元/分钟)': df_analysis['cost_per_min'].round(3)
})

print(analysis_summary.to_string(index=False))

# 计算整体统计
print(f"\n整体统计:")
print(f"平均距离: {(df['distance'].mean()/1000):.2f} 公里")
print(f"平均时长: {(df['duration'].mean()/60):.2f} 分钟")
print(f"平均成本: {df['cost'].mean():.2f} 元")
print(f"平均速度: {df_analysis['speed_kmh'].mean():.2f} km/h")
print(f"平均单位成本: {df_analysis['cost_per_km'].mean():.3f} 元/公里")

# 相关性分析
print(f"\n" + "="*60)
print("变量相关性分析")
print("="*60)
correlation_matrix = df[numeric_cols].corr()
print("相关系数矩阵:")
print(correlation_matrix.round(3))

# 分析相关性强度
print(f"\n相关性解读:")
for i in range(len(numeric_cols)):
    for j in range(i+1, len(numeric_cols)):
        var1 = numeric_cols[i]
        var2 = numeric_cols[j]
        corr_value = correlation_matrix.iloc[i, j]
        
        if abs(corr_value) >= 0.8:
            strength = "极强"
        elif abs(corr_value) >= 0.6:
            strength = "强"
        elif abs(corr_value) >= 0.4:
            strength = "中等"
        elif abs(corr_value) >= 0.2:
            strength = "弱"
        else:
            strength = "极弱"
        
        direction = "正" if corr_value > 0 else "负"
        print(f"- {var1} 与 {var2}: {direction}{strength}相关 (r = {corr_value:.3f})")

print(f"\n✅ 数据分析完成，准备生成最终报告")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 设置中文字体
plt.rcParams['font.sans-serif'] = ['WenQuanYi Zen Hei']
plt.rcParams['axes.unicode_minus'] = False

# 读取新数据文件
df_new = pd.read_csv('../data/Dimension-3.csv')

# 1. 数据基本信息探索
print("="*60)
print("1. 数据基本信息概览 (Dimension-3.csv)")
print("="*60)
print(f"数据形状: {df_new.shape} (行数: {df_new.shape[0]}, 列数: {df_new.shape[1]})")
print(f"\n数据类型分布:")
print(df_new.dtypes.value_counts())

print(f"\n列名列表:")
for i, col in enumerate(df_new.columns, 1):
    print(f"{i:2d}. {col}")

print(f"\n前5行数据预览:")
print(df_new.head())

# 2. 缺失值分析
print("\n" + "="*60)
print("2. 缺失值分析")
print("="*60)
missing_info = pd.DataFrame({
    '缺失数量': df_new.isnull().sum(),
    '缺失比例(%)': (df_new.isnull().sum() / len(df_new) * 100).round(2)
}).sort_values('缺失数量', ascending=False)

missing_info = missing_info[missing_info['缺失数量'] > 0]
if len(missing_info) > 0:
    print(missing_info)
else:
    print("✓ 数据无缺失值")

# 3. 数据统计描述
print("\n" + "="*60)
print("3. 数值型数据统计描述")
print("="*60)
numeric_cols_new = df_new.select_dtypes(include=[np.number]).columns
if len(numeric_cols_new) > 0:
    print(df_new[numeric_cols_new].describe().round(2))
else:
    print("数据中无数值型列")

# 4. 分类变量分析
print("\n" + "="*60)
print("4. 分类变量分析")
print("="*60)
categorical_cols_new = df_new.select_dtypes(include=['object', 'category']).columns
if len(categorical_cols_new) > 0:
    for col in categorical_cols_new:
        print(f"\n{col}:")
        print(f"  唯一值数量: {df_new[col].nunique()}")
        print(f"  所有唯一值: {list(df_new[col].unique())}")
        print(f"  各值分布:")
        print(df_new[col].value_counts())
else:
    print("数据中无分类变量")

# 5. 异常值检测（基于IQR方法）
print("\n" + "="*60)
print("5. 异常值检测（基于IQR方法）")
print("="*60)
outlier_summary_new = []
if len(numeric_cols_new) > 0:
    for col in numeric_cols_new:
        Q1 = df_new[col].quantile(0.25)
        Q3 = df_new[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df_new[(df_new[col] < lower_bound) | (df_new[col] > upper_bound)][col]
        outlier_count = len(outliers)
        outlier_ratio = round((outlier_count / len(df_new) * 100), 2)
        
        outlier_summary_new.append({
            '变量名': col,
            '异常值数量': outlier_count,
            '异常值比例(%)': outlier_ratio,
            '下界': round(lower_bound, 2),
            '上界': round(upper_bound, 2)
        })
    
    outlier_df_new = pd.DataFrame(outlier_summary_new)
    print("异常值汇总:")
    print(outlier_df_new.to_string(index=False))
else:
    print("无数值型变量可进行异常值检测")

print("\n" + "="*60)
print("数据初步探索完成，开始生成可视化图表")
print("="*60)

In [ ]:
# 生成综合可视化图表并进行深度分析
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 重新读取数据
df_new = pd.read_csv('../data/Dimension-3.csv')

# 计算衍生指标
df_analysis_new = df_new.copy()
# 单位转换
df_analysis_new['distance_km'] = df_analysis_new['distance'] / 1000  # 转换为公里
df_analysis_new['duration_min'] = df_analysis_new['duration'] / 60    # 转换为分钟
df_analysis_new['duration_hour'] = df_analysis_new['duration'] / 3600  # 转换为小时
# 计算速度和单位成本
df_analysis_new['speed_kmh'] = df_analysis_new['distance_km'] / df_analysis_new['duration_hour']  # 公里/小时
df_analysis_new['cost_per_km'] = df_analysis_new['cost'] / df_analysis_new['distance_km']  # 元/公里
df_analysis_new['cost_per_min'] = df_analysis_new['cost'] / df_analysis_new['duration_min']  # 元/分钟

# 创建综合可视化图表
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('不同交通方式出行数据分析报告 (Dimension-3.csv)', fontsize=20, fontweight='bold', y=0.98)

# 定义专业配色方案（按交通方式区分）
transport_colors = {
    '驾车': '#E74C3C',  # 红色
    '骑行': '#3498DB',  # 蓝色
    '步行': '#2ECC71'   # 绿色
}
transport_light_colors = {
    '驾车': '#FFE6E6',
    '骑行': '#E6F3FF',
    '步行': '#E6FFEE'
}

# 1. 各交通方式时长对比（主图）
ax1 = axes[0, 0]
transport_modes = df_analysis_new['transport_mode'].tolist()
durations_min = df_analysis_new['duration_min'].tolist()
colors_list = [transport_colors[mode] for mode in transport_modes]

# 创建柱状图
bars1 = ax1.bar(transport_modes, durations_min, color=colors_list, alpha=0.8, 
                edgecolor='white', linewidth=3, width=0.6)
ax1.set_xlabel('交通方式', fontsize=13, fontweight='bold')
ax1.set_ylabel('出行时长 (分钟)', fontsize=13, fontweight='bold')
ax1.set_title('不同交通方式出行时长对比\n（相同起点终点：25.81°N→25.74°N）', 
              fontsize=15, fontweight='bold', pad=20)
ax1.grid(True, alpha=0.3, linestyle='--', axis='y')

# 添加数值标签和时长单位转换
for bar, duration_min, duration_hour in zip(bars1, durations_min, df_analysis_new['duration_hour']):
    height = bar.get_height()
    # 显示分钟和小时两种单位
    label_text = f'{duration_min:.0f}分钟\n({duration_hour:.1f}小时)'
    ax1.text(bar.get_x() + bar.get_width()/2., height + 20,
             label_text, ha='center', va='bottom', 
             fontsize=11, fontweight='bold', linespacing=1.2)

# 2. 各交通方式成本对比
ax2 = axes[0, 1]
costs = df_analysis_new['cost'].tolist()

# 创建柱状图
bars2 = ax2.bar(transport_modes, costs, color=colors_list, alpha=0.8, 
                edgecolor='white', linewidth=3, width=0.6)
ax2.set_xlabel('交通方式', fontsize=13, fontweight='bold')
ax2.set_ylabel('出行成本 (元)', fontsize=13, fontweight='bold')
ax2.set_title('不同交通方式出行成本对比', fontsize=15, fontweight='bold', pad=20)
ax2.grid(True, alpha=0.3, linestyle='--', axis='y')

# 添加数值标签
for bar, cost in zip(bars2, costs):
    height = bar.get_height()
    # 根据成本值调整标签显示
    if cost == 0:
        label_text = '免费'
    else:
        label_text = f'{cost:.2f}元'
    ax2.text(bar.get_x() + bar.get_width()/2., height + 0.1,
             label_text, ha='center', va='bottom', 
             fontsize=12, fontweight='bold')

# 3. 各交通方式速度对比
ax3 = axes[1, 0]
speeds = df_analysis_new['speed_kmh'].tolist()

# 创建柱状图
bars3 = ax3.bar(transport_modes, speeds, color=colors_list, alpha=0.8, 
                edgecolor='white', linewidth=3, width=0.6)
ax3.set_xlabel('交通方式', fontsize=13, fontweight='bold')
ax3.set_ylabel('平均速度 (km/h)', fontsize=13, fontweight='bold')
ax3.set_title('不同交通方式平均速度对比', fontsize=15, fontweight='bold', pad=20)
ax3.grid(True, alpha=0.3, linestyle='--', axis='y')

# 添加数值标签
for bar, speed in zip(bars3, speeds):
    height = bar.get_height()
    ax3.text(bar.get_x() + bar.get_width()/2., height + 0.5,
             f'{speed:.1f} km/h', ha='center', va='bottom', 
             fontsize=12, fontweight='bold')

# 4. 综合指标雷达图数据准备（使用极坐标）
ax4 = axes[1, 1]

# 选择需要展示的指标（标准化处理，使其范围在0-100之间）
metrics = ['速度', '成本经济性', '耗时效率']
transport_data = {}

for _, row in df_analysis_new.iterrows():
    mode = row['transport_mode']
    # 速度标准化（最高速度为100分）
    speed_score = (row['speed_kmh'] / df_analysis_new['speed_kmh'].max()) * 100
    # 成本经济性（成本越低分数越高，免费为100分）
    cost_score = (1 - (row['cost'] / df_analysis_new['cost'].max())) * 100 if df_analysis_new['cost'].max() > 0 else 100
    # 耗时效率（时间越短分数越高）
    time_score = (1 - (row['duration_min'] / df_analysis_new['duration_min'].max())) * 100
    
    transport_data[mode] = [speed_score, cost_score, time_score]

# 创建雷达图（使用柱状图替代极坐标，更易读）
x = np.arange(len(metrics))
width = 0.25  # 柱子宽度

# 绘制各交通方式的指标柱
for i, (mode, scores) in enumerate(transport_data.items()):
    ax4.bar(x + i*width, scores, width, label=mode, color=transport_colors[mode], 
            alpha=0.7, edgecolor='white', linewidth=2)

# 设置雷达图样式
ax4.set_xlabel('评价指标', fontsize=13, fontweight='bold')
ax4.set_ylabel('标准化得分 (0-100分)', fontsize=13, fontweight='bold')
ax4.set_title('不同交通方式综合指标对比\n（分数越高越好）', fontsize=15, fontweight='bold', pad=20)
ax4.set_xticks(x + width)
ax4.set_xticklabels(metrics)
ax4.legend(loc='upper right', fontsize=11)
ax4.grid(True, alpha=0.3, linestyle='--', axis='y')
ax4.set_ylim(0, 105)  # 设置y轴范围

# 添加数值标签
for i, (mode, scores) in enumerate(transport_data.items()):
    for j, score in enumerate(scores):
        ax4.text(j + i*width, score + 2, f'{score:.0f}', 
                ha='center', va='bottom', fontsize=10, fontweight='bold')

# 调整布局
plt.tight_layout()
plt.subplots_adjust(top=0.93)

# 保存图表
plt.savefig('../data/transportation_analysis_visualization.png', dpi=300, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
plt.close()

# 生成详细分析表
print("="*80)
print("详细分析结果汇总表")
print("="*80)
analysis_summary_new = pd.DataFrame({
    '交通方式': df_analysis_new['transport_mode'],
    '起点坐标': df_analysis_new['start'],
    '终点坐标': df_analysis_new['destination'],
    '距离': [f"{km:.2f}公里" for km in df_analysis_new['distance_km']],
    '时长': [f"{min:.0f}分钟({hour:.1f}小时)" for min, hour in zip(df_analysis_new['duration_min'], df_analysis_new['duration_hour'])],
    '成本': [f"{cost:.2f}元" if cost > 0 else "免费" for cost in df_analysis_new['cost']],
    '平均速度': [f"{speed:.1f} km/h" for speed in df_analysis_new['speed_kmh']],
    '单位成本': [f"{cost_km:.3f}元/公里" if cost_km > 0 else "免费" for cost_km in df_analysis_new['cost_per_km']]
})

print(analysis_summary_new.to_string(index=False))

# 计算关键统计对比
print(f"\n" + "="*80)
print("关键指标对比分析")
print("="*80)

# 计算各方式相对于驾车的比例
driving_data = df_analysis_new[df_analysis_new['transport_mode'] == '驾车'].iloc[0]

for _, row in df_analysis_new.iterrows():
    mode = row['transport_mode']
    if mode == '驾车':
        continue
    
    time_ratio = (row['duration_min'] / driving_data['duration_min']) * 100
    cost_ratio = (row['cost'] / driving_data['cost'] * 100) if driving_data['cost'] > 0 else 0
    speed_ratio = (row['speed_kmh'] / driving_data['speed_kmh']) * 100
    
    print(f"\n{mode} vs 驾车:")
    print(f"  - 时长比例: {time_ratio:.0f}% (是驾车的{time_ratio/100:.1f}倍)")
    print(f"  - 成本比例: {cost_ratio:.1f}% (仅为驾车的{cost_ratio/100:.1f}倍)")
    print(f"  - 速度比例: {speed_ratio:.0f}% (仅为驾车的{speed_ratio/100:.1f}倍)")

# 相关性分析
print(f"\n" + "="*80)
print("变量相关性分析")
print("="*80)
numeric_cols_new = ['distance', 'duration', 'cost']
correlation_matrix_new = df_analysis_new[numeric_cols_new].corr()
print("相关系数矩阵:")
print(correlation_matrix_new.round(3))

# 分析相关性强度
print(f"\n相关性解读:")
for i in range(len(numeric_cols_new)):
    for j in range(i+1, len(numeric_cols_new)):
        var1 = numeric_cols_new[i]
        var2 = numeric_cols_new[j]
        corr_value = correlation_matrix_new.iloc[i, j]
        
        if abs(corr_value) >= 0.8:
            strength = "极强"
        elif abs(corr_value) >= 0.6:
            strength = "强"
        elif abs(corr_value) >= 0.4:
            strength = "中等"
        elif abs(corr_value) >= 0.2:
            strength = "弱"
        else:
            strength = "极弱"
        
        direction = "正" if corr_value > 0 else "负"
        print(f"- {var1} 与 {var2}: {direction}{strength}相关 (r = {corr_value:.3f})")

print(f"\n✅ 数据分析完成，可视化图表已生成")